# Exploración de señales de riesgo — Radar de Contratación Pública

**Semana 3 (Data Science).** Recorrido por las señales estadísticas de la capa
`analytics/` sobre los marts de dbt (`main.fct_adjudicaciones`).

> ⚠️ **Aviso**: son *señales a revisar*, nunca acusaciones. Análisis agregado, con
> métodos robustos, sobre datos abiertos oficiales. Cumplimiento RGPD por diseño.


In [ ]:
import pandas as pd

from analytics import anomalias, importe
from analytics.riesgo import informe_riesgo

pd.set_option("display.max_colwidth", 60)

## 1. Informe consolidado

`informe_riesgo()` compone las tres señales y ranquea los órganos por número de
indicios acumulados (baja atípica · concentración · exceso de oferta única).

In [ ]:
inf = informe_riesgo()
print("órganos con al menos una señal:", len(inf.organos))
print("reparto por nº de señales:",
      inf.organos["n_senales"].value_counts().sort_index(ascending=False).to_dict())
inf.organos.head(10)[
    ["organo_contratacion", "n_senales", "n_bajas_atipicas",
     "hhi", "cuota_dominante", "tasa_oferta_unica"]
]

## 2. Baja atípica por mercado (z-score robusto)

Dentro de cada `cpv_division × tipo_contrato`, la baja de cada adjudicación se compara
con la mediana del grupo vía z robusto (MAD). Se marcan ambas colas: baja ~0
(adjudicación pegada al presupuesto) y baja extrema (posible temeridad).

In [ ]:
ba = anomalias.baja_atipica()
print("adjudicaciones atípicas:", len(ba))
print(ba["cola"].value_counts().to_dict())
ba.head(8)[["organo_contratacion", "grupo", "baja_pct", "z_robusto", "cola", "n_grupo"]]

## 3. Concentración por órgano (HHI)

Índice Herfindahl-Hirschman del reparto del importe entre adjudicatarios de cada
órgano. Referencia antitrust: <1500 competido · 1500–2500 moderado · >2500 concentrado.

In [ ]:
conc = anomalias.concentracion_organo()
print("niveles:", conc["nivel"].value_counts().to_dict())
conc.head(8)[["organo_contratacion", "n_adjudicaciones", "n_adjudicatarios",
              "hhi", "cuota_dominante", "adjudicatario_dominante", "nivel"]]

## 4. Exceso de oferta única (test de proporción)

Órganos cuya tasa de adjudicaciones con un único licitador supera el promedio del
sistema de forma estadísticamente significativa (z de proporción, cola derecha).

In [ ]:
ofu = anomalias.exceso_oferta_unica()
print("tasa global de oferta única (p0):", round(ofu.attrs["p0_global"], 3))
print("órganos significativos:", int(ofu["significativo"].sum()), "de", len(ofu))
ofu.head(8)[["organo_contratacion", "n", "tasa", "p_valor", "significativo"]]

## 5. Baja esperada con incertidumbre

Dos lecturas de la incertidumbre de la baja de un contrato a partir de features
pre-adjudicación (CPV, tipo, procedimiento, tamaño del presupuesto).

### 5.1 Regresión cuantílica (interpretable)

`QuantReg` en p10/p50/p90. Los coeficientes dicen qué mueve la baja.

In [ ]:
modelo = importe.ajustar_cuantilica()
modelo.resumen_coeficientes(0.5).head(8)

### 5.2 CQR — intervalos con cobertura garantizada

Conformalized Quantile Regression: calibra el intervalo para garantizar cobertura
marginal ≥ 1−α y la mide en un test hold-out independiente.

In [ ]:
r = importe.intervalos_conformal(alpha=0.1)
print(f"objetivo de cobertura : {r.cobertura_objetivo:.0%}")
print(f"cobertura empírica    : {r.cobertura_empirica:.1%}")
print(f"anchura media         : {r.anchura_media:.3f}")
print(f"corrección conformal  : {r.correccion:.3f}")
r.predicciones.head()